# Importing and cleaning the air pollution data

In [0]:
import os
import glob
from pyspark.sql.functions import col, to_timestamp, regexp_replace

current_dir = os.getcwd() 
raw_path = os.path.join(current_dir, "raw_data")
no2_files = glob.glob(os.path.join(raw_path, "*NO2*.csv"))

df_final = None

for file_path in no2_files:
    spark_file_path = f"file:{file_path}"
    
    df_raw = spark.read.option("header", "false") \
                       .option("delimiter", ";") \
                       .option("encoding", "windows-1250") \
                       .csv(spark_file_path)
                       
    sample_rows = df_raw.limit(15).collect()
    
    target_col = None
    for row in sample_rows:
        row_dict = row.asDict()
        for col_name, val in row_dict.items():
            if val and "MzWarAlNiepo" in str(val):
                target_col = col_name
                break
        if target_col:
            break
            
    if target_col:
        df_filtered = df_raw.select(
            col("_c0").alias("Data_Czas"),
            col(target_col).alias("NO2_Wartosc")
        )
        
        df_clean = df_filtered.filter(col("Data_Czas").rlike(r'^\d{2}\.\d{2}\.\d{4}'))
        
        if df_final is None:
            df_final = df_clean
        else:
            df_final = df_final.union(df_clean)

if df_final is not None:
    silver_no2 = df_final \
        .withColumn("NO2_Wartosc", regexp_replace(col("NO2_Wartosc"), ",", ".").cast("float")) \
        .withColumn("Data_Czas", to_timestamp(col("Data_Czas"), "dd.MM.yyyy HH:mm")) \
        .dropna(subset=["NO2_Wartosc"])

    print(f"Sukces! Całkowita liczba rekordów (Pure PySpark): {silver_no2.count()}")
    
    silver_no2.write.format("delta").mode("overwrite").saveAsTable("silver_no2_warsaw")
    print("Zapisano tabelę silver_no2_warsaw w formacie Delta!")
    display(silver_no2)
else:
    print("Nie udało się odnaleźć stacji w plikach.")

Sukces! Całkowita liczba rekordów (Pure PySpark): 86779
Zapisano tabelę silver_no2_warsaw w formacie Delta!


Data_Czas,NO2_Wartosc
2015-01-01T01:00:00.000Z,25.0
2015-01-01T02:00:00.000Z,28.0
2015-01-01T03:00:00.000Z,28.0
2015-01-01T04:00:00.000Z,28.0
2015-01-01T05:00:00.000Z,26.0
2015-01-01T06:00:00.000Z,25.0
2015-01-01T07:00:00.000Z,22.0
2015-01-01T08:00:00.000Z,17.0
2015-01-01T09:00:00.000Z,16.0
2015-01-01T10:00:00.000Z,14.0


# Conversion test

In [0]:
from pyspark.sql.functions import min, max

total_rows = silver_no2.count()
print(f"Całkowita liczba rekordów w bazie: {total_rows}")

silver_no2.select(min("Data_Czas").alias("Najstarszy_pomiar"), max("Data_Czas").alias("Najnowszy_pomiar")).show()

display(silver_no2.orderBy("Data_Czas"))

Całkowita liczba rekordów w bazie: 86779
+-------------------+-------------------+
|  Najstarszy_pomiar|   Najnowszy_pomiar|
+-------------------+-------------------+
|2015-01-01 01:00:00|2025-01-01 00:00:00|
+-------------------+-------------------+



Data_Czas,NO2_Wartosc
2015-01-01T01:00:00.000Z,25.0
2015-01-01T02:00:00.000Z,28.0
2015-01-01T03:00:00.000Z,28.0
2015-01-01T04:00:00.000Z,28.0
2015-01-01T05:00:00.000Z,26.0
2015-01-01T06:00:00.000Z,25.0
2015-01-01T07:00:00.000Z,22.0
2015-01-01T08:00:00.000Z,17.0
2015-01-01T09:00:00.000Z,16.0
2015-01-01T10:00:00.000Z,14.0


In [0]:
import os
from pyspark.sql.functions import col, round

current_dir = os.getcwd() 
raw_path = os.path.join(current_dir, "raw_data")
cars_file_path = f"file:{os.path.join(raw_path, 'cars_warsaw_2015-2024.csv')}"

df_cars_raw = spark.read.option("header", "true") \
                        .option("delimiter", ";") \
                        .option("encoding", "utf-8") \
                        .option("quote", '"') \
                        .csv(cars_file_path)

df_cars_raw.createOrReplaceTempView("raw_cars")

queries = []
for rok in range(2015, 2025):
    q = f"""
    SELECT 
        {rok} as Rok,
        CAST(`samochody osobowe;ogółem;{rok};[szt.]` AS INT) as Osobowe_Ogolem,
        CAST(`samochody osobowe;pozostałe;{rok};[szt.]` AS INT) as Osobowe_Pozostale
    FROM raw_cars
    """
    queries.append(q)

final_query = " UNION ALL ".join(queries)
silver_cars = spark.sql(final_query)

silver_cars = silver_cars.withColumn(
    "Procent_Nowoczesnej_Floty", 
    round((col("Osobowe_Pozostale") / col("Osobowe_Ogolem")) * 100, 2)
)

silver_cars.write.format("delta").mode("overwrite").saveAsTable("silver_cars_warsaw")
print("Tabela Aut wyczyszczona i gotowa!")
display(silver_cars)

Tabela Aut wyczyszczona i gotowa!


Rok,Osobowe_Ogolem,Osobowe_Pozostale,Procent_Nowoczesnej_Floty
2015,1131120,7533,0.67
2016,1194068,9405,0.79
2017,1261803,15372,1.22
2018,1332923,26033,1.95
2019,1393218,44126,3.17
2020,1425475,61706,4.33
2021,1460479,92230,6.32
2022,1518315,127818,8.42
2023,1574854,168607,10.71
2024,1336242,184940,13.84
